In [ ]:
upstream = None 
product = None
mcleish24_tables1 = None
mcleish24_tables2 = None

# Experimental data: Sites and libraries

In [ ]:
import pandas as pd
import json

## Load data

In [ ]:
libraries = pd.read_csv(mcleish24_tables2, sep=';')
sampling = pd.read_csv(mcleish24_tables1, sep=';')


Make lists out of dates. 

In [ ]:
sampling = sampling.groupby(['Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude'], as_index = False)['Date'].apply(list)

## Connecting Library codes with Site codes

We have a small issue: The original data contains "library codes", "site codes" and "collection codes". So far, we don't care about collection codes but about "Site" codes. Therefore, we are going to remove such a dependency by crossing dataframes.

Separate collections by '_' (some libraries belong to more than one collection)

In [ ]:
libraries['Collection_code'] = libraries['Collection_code'].apply(lambda x: x.split("_"))

We make dataframe that helps us to connect collections and libraries. 

In [ ]:
tmp = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'Collection_code']].to_dict(orient='records')
library_collection_code = []
for item in tmp:
    for collection_code in item['Collection_code']:
        library_collection_code.append({"Library_code": item['Library_code'], "Collection_code": collection_code})
library_collection_code = pd.DataFrame.from_records(library_collection_code)
library_collection_code

We use the previous dataframe and we combine it with library information, to connect Library code with Site code. 

In [ ]:
library_site = pd.merge(sampling[['Collection_code', 'Site_code']], library_collection_code, on='Collection_code')
library_site = library_site[['Site_code', 'Library_code']].to_dict(orient='records')


## Creating a separate dates table

We will use this separate table to relate libraries and their collection dates.

In [ ]:
tmp = pd.merge(sampling, library_collection_code, on='Collection_code')
tmp = tmp.drop_duplicates(subset=['Library_code'])[['Library_code', 'Date']].to_dict(orient='records')
dates = []
for item in tmp:
    for date in item['Date']:
        dates.append({"Library_code": item['Library_code'], "Date": date})
dates

## Modeling sites

We will now create a table with all site metadata, such as habitat, location, etc. 

This labels were proposed by Mark. I am not sure how to use them for anything.

In [ ]:
sites_to_type = {
    "Crop": "CO_020:0000051",
    "Edge": "CO_020:0000054",
    "Wasteland": "CO_020:0000046",
    "Oak": "CO_020:0000045"
}

Merging site data with libary codes

In [ ]:
sites = pd.merge(
    pd.merge(sampling, library_collection_code, on='Collection_code'),
    libraries[['Library_code', 'Habitat']], on='Library_code'
)
sites['Habitat_type'] = sites['Habitat'].map(sites_to_type)
sites = sites.drop_duplicates(subset=['Site_code'])[['Site_code', 'Longitude', 'Latitude', 'Location', 'Habitat', 'Habitat_type']].to_dict(orient='records')
sites

## Libraries table

Libraries are very easy to model here: we only need to consider their codes and the number of samples that they are associated with. 

In [ ]:
libraries = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'No_extracts']].to_dict(orient='records')
libraries

## Final output

In [ ]:
site_library = dict(
    library=libraries,
    library_dates=dates,
    site=sites, 
    library_site=library_site
)
with open(product['data'], "w") as f:
    json.dump(site_library, f, indent=4, sort_keys=True)